In [1]:
pip install langchain_google_genai

In [2]:
#Import packages
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

#Get the APIkey
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(
    google_api_key=GOOGLE_API_KEY, model = "gemini-2.5-flash")

In [3]:
response = llm.invoke([HumanMessage(content="Hello")])
print(response.content)

Hello! How can I help you today?


Lang Chain Ingestion and Retrieval Workflow

In [4]:
pip install -U chromadb tiktoken

In [5]:
pip install -U langchain-community

In [6]:
hr_policy_text = """
Employee Code of Conduct & Workplace Policy
This document outlines the core standards expected of all employees to ensure a professional, safe, and productive workplace.

1. Equal Opportunity and Anti-Harassment
We are committed to a work environment free from discrimination and harassment. We do not tolerate any form of discrimination based on race, gender, religion, age, disability, or any other protected characteristic. Harassment or bullying of any kind is strictly prohibited and will result in immediate disciplinary action.

2. Professional Conduct and Confidentiality
Employees are expected to act with integrity and professionalism at all times. You are required to protect company information, including client data, trade secrets, and internal processes. Maintaining confidentiality is a condition of your employment and continues even if your employment ends.

3. Attendance and Punctuality
Reliability is essential to our collective success. Employees are expected to be present during their scheduled hours and to provide advance notice for any planned absences or leaves of absence in accordance with our time-off procedures. Unplanned absences should be communicated to your direct supervisor as soon as possible.

4. Workplace Safety and Health
The safety of our employees is our top priority. All staff must comply with established health and safety protocols, including maintaining a clean workspace and reporting any hazards or incidents immediately. We are committed to providing an environment that adheres to all local occupational health and safety regulations.

5. Use of Company Property
Company assets—including technology, equipment, and intellectual property—are provided to assist in the performance of your job duties. Employees are responsible for the proper care of these assets and must use them in a manner that is ethical and aligned with company security policies. Personal use of company equipment should be minimal and reasonable.

"""
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document

documents = [Document(page_content=hr_policy_text, metadata = {"source": "HR Policy Handbook"})]

print(f"Loaded {len(documents)} documents")
print('First 250 characters of the document:')
print(documents[0].page_content[:250])

/tmp/ipykernel_19438/805491190.py:21: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Loaded 1 documents
First 250 characters of the document:

Employee Code of Conduct & Workplace Policy
This document outlines the core standards expected of all employees to ensure a professional, safe, and productive workplace.

1. Equal Opportunity and Anti-Harassment
We are committed to a work environmen


Split document into chunks


In [7]:
pip install -U langchain-text-splitters

In [8]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200,
    length_function = len,
    add_start_index = True,
)

chunks = text_splitter.split_documents(documents)

print(f"Split document into {len(chunks)} chunks")
print("First chunk:")
print(chunks[0].page_content)

Split document into 3 chunks
First chunk:
Employee Code of Conduct & Workplace Policy
This document outlines the core standards expected of all employees to ensure a professional, safe, and productive workplace.

1. Equal Opportunity and Anti-Harassment
We are committed to a work environment free from discrimination and harassment. We do not tolerate any form of discrimination based on race, gender, religion, age, disability, or any other protected characteristic. Harassment or bullying of any kind is strictly prohibited and will result in immediate disciplinary action.

2. Professional Conduct and Confidentiality
Employees are expected to act with integrity and professionalism at all times. You are required to protect company information, including client data, trade secrets, and internal processes. Maintaining confidentiality is a condition of your employment and continues even if your employment ends.


Initialize Google Generate Ai Embedding

In [9]:
from google.colab import userdata
from langchain_google_genai import GoogleGenerativeAIEmbeddings

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
embeddings = GoogleGenerativeAIEmbeddings(
    google_api_key=GOOGLE_API_KEY, model="models/embedding-001"
)

print("Google Generative AI Embedding Initialized")

Google Generative AI Embedding Initialized


Create and Populate Chroma Vector Store

In [10]:
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

embeddings = GoogleGenerativeAIEmbeddings(
    model='models/gemini-embedding-001', google_api_key=GOOGLE_API_KEY)

vectorstore = Chroma.from_documents(chunks, embeddings)

print(f"Chroma vector store created with {vectorstore._collection.count()} entries.")

retriever = vectorstore.as_retriever()

print("Retriever created from Chroma vector store.")

Chroma vector store created with 3 entries.
Retriever created from Chroma vector store.


Set up the RAG chain


In [20]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
    Answer the question based only on the following context:
    {context}

    Question: {input}
    If you do not know the answer, just say that you don't know. Don't try to make up an answer.

    """
)

class CustomRetrievalChain:
  def invoke(self, input_dict):
    question = input_dict["input"]

    retrieved_docs = retriever.invoke(question)

    formatted_context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    formatted_prompt_messages = prompt.format_messages(
        context=formatted_context,
        input=question)

    response = llm.invoke(formatted_prompt_messages)

    return {"Answer": response.content}

retrieval_chain = CustomRetrievalChain()

print("RAG 'retrival_chain' (manual implementation) created sucessfully.")

RAG 'retrival_chain' (manual implementation) created sucessfully.


Demo Retrieval


In [21]:
question = "What is the policy for company property?"
print(f"\nQuestion: {question}")
response = retrieval_chain.invoke({"input":question})
print(f"\nAnswer: {response['Answer']}")

question_2 = "What is the policy taking leaves?"
print(f"\nQuestion: {question_2}")
response_2 = retrieval_chain.invoke({"input":question_2})
print(f"\nAnswer: {response_2['Answer']}")

question_3 = "What is the rule on harrassment?"
print(f"\nQuestion: {question_3}")
response_3 = retrieval_chain.invoke({"input":question_3})
print(f"\nAnswer: {response_3['Answer']}")


Question: What is the policy for company property?

Answer: Company assets, including technology, equipment, and intellectual property, are provided to assist employees in performing their job duties. Employees are responsible for the proper care of these assets and must use them ethically and in alignment with company security policies. Personal use of company equipment should be minimal and reasonable.

Question: What is the policy taking leaves?

Answer: Employees are expected to provide advance notice for any planned absences or leaves of absence in accordance with the company's time-off procedures. Unplanned absences should be communicated to their direct supervisor as soon as possible.

Question: What is the rule on harrassment?

Answer: Harassment or bullying of any kind is strictly prohibited and will result in immediate disciplinary action.
